In [155]:
import pandas as pd

# Importa a base de dados
caminho = r'C:\Users\Adm\Downloads\2025-10-01_Alvaras_-_Base_de_Dados.csv'

# Leitura do CSV
df = pd.read_csv(
    caminho,
    sep=';',                # separador usado em CSVs brasileiros
    encoding='latin1',      
    quotechar='"',        
    dtype=str               # lê tudo como texto (útil para limpeza)
)

# Mostra informações iniciais
print(df.shape)
df.head()

(512651, 213)


,NOME_EMPRESARIAL,INICIO_ATIVIDADE,NUMERO_DO_ALVARA,NOME_FANTASIA,DATA_EMISSAO,DATA_EXPIRACAO,ENDERECO,NUMERO,UNIDADE,ANDAR,...,CNAE_ATIVIDADE_SECUNDARIA95,ATIVIDADE_SECUNDARIA95,CNAE_ATIVIDADE_SECUNDARIA96,ATIVIDADE_SECUNDARIA96,CNAE_ATIVIDADE_SECUNDARIA97,ATIVIDADE_SECUNDARIA97,CNAE_ATIVIDADE_SECUNDARIA98,ATIVIDADE_SECUNDARIA98,CNAE_ATIVIDADE_SECUNDARIA99,ATIVIDADE_SECUNDARIA99
0,H F A CORRETORA DE SEGUROS S/C LTDA,15/01/1996,650161,***,21/07/2004,***,R. CARLOS ESSENFELDER,003113,***,***,...,***,***,***,***,***,***,***,***,***,***
1,INSTITUTO BENEFICIENTE AGUAS CRISTALINAS,26/05/2009,925073,IBAC,03/03/2010,***,R. ANNE FRANK,005059,***,***,...,***,***,***,***,***,***,***,***,***,***
2,FRANCISCA LUCINEUMA DE SOUSA 85754650906,03/08/2017,1331619,SFERA,18/09/2017,***,R. PAULA PREVEDELLO GUSSO,001602,***,***,...,***,***,***,***,***,***,***,***,***,***
3,HIGHER ENERGY CONSULTORIA EM ENERGIAS RENOVAVE...,14/03/2016,1262050,HIGHER ENERGY CONSULTORIA EM ENERGIAS RENOVAVE...,16/03/2016,31/12/2017,AV. SETE DE SETEMBRO,004751,01,SB,...,***,***,***,***,***,***,***,***,***,***
4,FELIPE FLORES DE PRA 00720066921,25/04/2019,1443375,FELIPE FLORES DE PRA,15/07/2019,***,R. SANTA MARIA,000624,NaN,NaN,...,***,***,***,***,***,***,***,***,***,***


In [157]:
import re

# Exclui todas as colunas que não serão utilizadas
# Lista para armazenar as colunas a excluir
colunas_excluir = []

for col in df.columns:
    # Remove todas as colunas que começam com CNAE_ATIVIDADE_SECUNDARIA
    if col.upper().startswith('CNAE_ATIVIDADE_SECUNDARIA'):
        colunas_excluir.append(col)
    # Remove somente ATIVIDADE_SECUNDARIA06 até ATIVIDADE_SECUNDARIA99
    elif re.match(r'^ATIVIDADE_SECUNDARIA(0?[6-9]|[1-9][0-9])$', col.upper()):
        colunas_excluir.append(col)

# Remove as colunas selecionadas
df = df.drop(columns=colunas_excluir, errors='ignore')

print(f"{len(colunas_excluir)} colunas removidas:")
# print(colunas_excluir)

193 colunas removidas:


In [159]:
if "NUMERO" in df.columns:
    # Converter para string para garantir que os métodos de texto funcionem
    df["NUMERO"] = df["NUMERO"].astype(str)
    
    # Selecionar linhas onde o número NÃO é "000000"
    mask = df["NUMERO"] != "000000"
    
    # Remover zeros à esquerda apenas nessas linhas
    df.loc[mask, "NUMERO"] = (
        df.loc[mask, "NUMERO"]
        .str.lstrip("0")
        .replace("", "0")
    )

Limpeza de zeros à esquerda concluída (mantendo '000000' intacto).


In [161]:
# Lista das colunas que devem ser removidas
colunas_para_remover = ["UNIDADE", "ANDAR", "COMPLEMENTO"]

df = df.drop(columns=colunas_para_remover, errors='ignore')

Colunas UNIDADE, ANDAR e COMPLEMENTO removidas com sucesso.


In [163]:
# Mostra todos os dados únicos de atividade principal

valores_distintos = df['ATIVIDADE_PRINCIPAL'].unique()
#print(list(valores_distintos))

# Mostra a quantidade de atividades principais diferentes
num_distintos = df['ATIVIDADE_PRINCIPAL'].nunique()
print(num_distintos)
print(len(df))

10702
512651


In [165]:
# Lista de prefixos
prefixos_sacolao = [
    "SACOL", "HORT", "ORTF",
    "VERDURA", "FRUT", "LEGUM",
    "QUITAND", "SUPERMERCADO"
]

# Cria expressão regex combinando todos os prefixos (case-insensitive)
regex = "|".join(prefixos_sacolao)

# Filtra as atividades que contêm qualquer um dos prefixos
df_filtrado = df[df["ATIVIDADE_PRINCIPAL"].str.contains(regex, case=False, na=False)]

# Extrai apenas os valores únicos da coluna ATIVIDADE_PRINCIPAL
atividades_unicas = df_filtrado["ATIVIDADE_PRINCIPAL"].drop_duplicates().sort_values()

#print(list(atividades_unicas))

contagem = len(atividades_unicas)
contagem_df = len(df_filtrado)

print(f"Número de atividades únicas encontradas: {contagem}")
print(f"Número de linhas: {contagem_df}")

Número de atividades únicas encontradas: 251
Número de linhas: 2091


In [167]:
atividades_para_excluir = [
    "COMÉRCIO ATACADISTA DE ALIMENTOS PARA ANIMAIS",
    "COMÉRCIO ATACADISTA DE CEREAIS E LEGUMINOSAS BENEFICIADOS",
    "COMÉRCIO ATACADISTA DE PESCADOS E FRUTOS DO MAR", 
    "FABR BOLSAS SACOLAS E CAPANGAS",
    "FABRIC BOLSAS SACOL MOCHILAS",
    "FABRICAÇÃO DE BOLSAS SACOLA MOCHILAS",
    "FABRICAÇÃO DE FRUTAS CRISTALIZADAS, BALAS E SEMELHANTES", 
    "FABRICAÇÃO DE CONSERVAS DE LEGUMES E OUTROS VEGETAIS, EXCETO PALMITO", 
    "FABRICACAO DE BOLSAS SACOLA MOCHI", 
    "IND BOLSAS SACOL MOCHILAS PAST ES",
    "IND BOLSAS SACOLAS MOCHILAS",
    "INDUSTRIA DE SACOLAS PLASTICAS", 
    "HORTICULTURA, EXCETO MORANGO",
    "LJ COM VAR DOCES EMB HORT",
    "LJ COM VAR RACOES E PROD HORTIGRA",
    "LJ COM VAR IMP EXP PEIX FRUT MAR",
    "LJ COM VAR ARM E QUITANDA",
    "LJ COM VAR PEIXES E FRUTOS DO MAR",
    "LJ DE COM MUDAS FRUTIF FLOR ORNAM",
    "LOJA COM PESCADOS E FRUTOS DO MAR",
    "MERCEARIA COM ART VEST QUITANDA",
    "PEIXARIA COM VRJ DE FRUTOS DO MAR",
    "LJ COM VAR PROD ALIM PEIX FRUTOS"
]

df_filtrado = df_filtrado[~df_filtrado['ATIVIDADE_PRINCIPAL'].isin(atividades_para_excluir)]

atividades_unicas = df_filtrado["ATIVIDADE_PRINCIPAL"].drop_duplicates().sort_values()

#print(list(atividades_unicas))

lista_atividades = list(atividades_unicas)

#print("Lista de atividades únicas:")
#for atividade in lista_atividades:
#    print(atividade)


contagem = len(atividades_unicas)

print(f"Número de atividades únicas encontradas: {contagem}")
print(len(df_filtrado))

Número de atividades únicas encontradas: 230
1946


In [169]:
palavras_excluir_nome = [
    "LANCH", 
    "DEPOSIT", "PIZZA", "RESTAURANTE",
    "PEIXARIA", "FESTAS", "DISTRIBUID", 
    "SORVETERIA", "MINIMERCADO", "MINI MERCADO"
]

regex_excluir_nome = "|".join(palavras_excluir_nome)

mascara_excluir_nome = (
    df_filtrado["NOME_FANTASIA"].str.contains(regex_excluir_nome, case=False, na=False)
    | df_filtrado["NOME_EMPRESARIAL"].str.contains(regex_excluir_nome, case=False, na=False)
)
df_excluidos = df_filtrado[mascara_excluir_nome].copy()
df_filtrado = df_filtrado[~mascara_excluir_nome]
print(f"Quantidade excluída: {len(df_excluidos)}")
#display(df_excluidos.head())
print(len(df_filtrado))

Quantidade excluída: 46
1900


In [171]:
atividades_unicas = df_filtrado["ATIVIDADE_PRINCIPAL"].drop_duplicates().sort_values()

print("Lista de atividades únicas:")
for atividade in lista_atividades:
    print(atividade)

contagem = len(atividades_unicas)

print(f"Número de atividades únicas encontradas: {contagem}")

Lista de atividades únicas:
ACOUG POST VEND PAES COM VAR HORT
ACOUGUE COM VAR HORTIFRUTIGR
ACOUGUE COM VAR HORTIFRUTIGRAN BE
ACOUGUE E QUITANDA
ACOUGUE E SUPERMERCADO
ACOUGUE MERC QUITANDA E ASSADOS
ACOUGUE MERCEARIA E QUITANDA
ACOUGUE PEIXARIA E QUITANDA
ACOUGUE QUITAND IMP EXP PROD ALIM
ACOUGUE QUITANDA COM DE CARVAO
ACOUGUE QUITANDA E COM CARNE AS
ACOUGUE QUITANDA E COM CARNE ASSA
BANCA COM HORTIFRUTIGRANJEIROS
BANCA COMERCIO FRUTAS E VERDURAS
BANCA COMERCIO VAREJO HORTIGRANJE
BANCA DE COM OVOS FRUT VERD E LEG
BAR MERCEARIA E QUITANDA
BAR QUITANDA E COM VAR CARNES AS
BC COM VAR PROD HORTIFRUTIGRANJEI
BOX CM FRUTAS E VERDU E PROD AGRO
BOX COM A VAREJO FRUTAS E VERDURA
BOX COM ATAC FRUT CER E SERV TRAN
BOX COM ATAC FRUTAS E VERD
BOX COM ATAC FRUTAS VERDURAS
BOX COM ATAC HORTIFRUT
BOX COM ATAC HORTIFRUTIGRANJEIROS
BOX COM ATAC IMP EXP PROD HORT
BOX COM ATAC PROD HORT VERD FRUTA
BOX COM ATAC PROD HORTIGRANJ
BOX COM ATAC PROD HORTIGRANJEIROS
BOX COM ATACAD DE CEREAIS E HORTI
BOX COM ATAC

In [173]:
palavras_excluir_atividade_principal = [
    "IND", "LANCHONETE", 
    "DEPOSIT", "REST",  
     "FABRIC", "ESCR"
]

# Cria regex para exclusão
regex_excluir = "|".join(palavras_excluir_atividade_principal)

# Exclui as atividades que contêm qualquer uma das substrings indesejadas
df_filtrado = df_filtrado[~df_filtrado["ATIVIDADE_PRINCIPAL"].str.contains(regex_excluir, case=False, na=False)]

# Extrai apenas os valores únicos da coluna ATIVIDADE_PRINCIPAL
atividades_unicas = df_filtrado["ATIVIDADE_PRINCIPAL"].drop_duplicates().sort_values()

df_filtrado = df_filtrado[df_filtrado["DATA_EXPIRACAO"] == "***"]

# Mostra a contagem final e os nomes das atividades
print(f"Número de atividades únicas encontradas: {len(atividades_unicas)}")
print(f"Número de alvarás únicos: {len(df_filtrado)}")
#print(list(atividades_unicas))

Número de atividades únicas encontradas: 212
Número de alvarás únicos: 1668


In [177]:
tem_panif = df_filtrado["NOME_EMPRESARIAL"].str.contains("PANIF", case=False, na=False)
tem_merc = df_filtrado["NOME_EMPRESARIAL"].str.contains("MERC", case=False, na=False)

# Exclui as linhas que têm "PANIF" mas não têm "MERC"
df_filtrado = df_filtrado[~(tem_panif & ~tem_merc)]

In [179]:
atividades_unicas = df_filtrado["ATIVIDADE_PRINCIPAL"].drop_duplicates().sort_values()

contagem = len(atividades_unicas)
alvaras_unicos = len(df_filtrado)

print(f"Número de atividades únicas encontradas: {contagem}")
print(f"Número de alvarás únicos encontrados: {alvaras_unicos}")

#print(list(atividades_unicas))

Número de atividades únicas encontradas: 173
Número de alvarás únicos encontrados: 1667


In [181]:
contagem = (
    df_filtrado
    .groupby("ATIVIDADE_PRINCIPAL")
    .size()
    .reset_index(name="QUANTIDADE")
    .sort_values(by="QUANTIDADE", ascending=False)
)

print(contagem)

                                   ATIVIDADE_PRINCIPAL  QUANTIDADE
114         COMÉRCIO VAREJISTA DE HORTIFRUTIGRANJEIROS         642
109  COMERCIO ATACADISTA DE FRUTAS, VERDURAS, RAÍZE...         504
115  COMÉRCIO VAREJISTA DE MERCADORIAS EM GERAL, CO...         205
152                               MERCEARIA E QUITANDA          32
159                                           QUITANDA          31
..                                                 ...         ...
61                        BOX DE COM ATAC DE HORTIFRUT           1
63                     BOX DE COM ATAC FRUTAS VERDURAS           1
64                   BOX DE COM ATAC HORTIFRUTIGRANJEI           1
65                     BOX DE COM ATAC HORTIGRANJEIROS           1
172                         VENDA DE VERDURAS A VAREJO           1

[173 rows x 2 columns]


In [183]:
import re
import numpy as np

def remover_zeros_endereco(df):
    if "NUMERO" in df.columns:
        df["NUMERO"] = df["NUMERO"].apply(
            lambda x: str(int(x)) if str(x).isdigit() and int(x) != 0 else np.nan
        )

    return df

df_filtrado = remover_zeros_endereco(df_filtrado)

In [185]:
df_filtrado["CEP"] = df_filtrado["CEP"].astype(str).str.strip()
df_filtrado["ENDERECO"] = df_filtrado["ENDERECO"].astype(str).str.upper().str.strip()

df_filtrado.loc[
    (df_filtrado["CEP"] == "80060090") & 
    (df_filtrado["ENDERECO"] == "AVENIDA PRESIDENTE AFFONSO CAMARGO"),
    "BAIRRO"
] = "JARDIM BOTANICO"

In [187]:
import re

def corrigir_1_maio(df):
    def corrigir_texto(x):
        if isinstance(x, str):
            return re.sub(
                r"RUA\s+1.*?DE\s+MAIO",
                "RUA PRIMEIRO DE MAIO",
                x,
                flags=re.IGNORECASE
            )
        return x

    for col in ["ENDERECO_COMPLETO", "ENDERECO"]:
        if col in df.columns:
            df[col] = df[col].apply(corrigir_texto)

    return df

df_filtrado = corrigir_1_maio(df_filtrado)

In [1122]:
# Definindo as coordenadas exatas da CEASA/BR-116 Tatuquara
lat_ceasa = -25.5566
lon_ceasa = -49.2785

dataframes = [df_filtrado]

for df in dataframes:
    mask = (
        (df["NUMERO"].astype(str) == "22881") & 
        (df["BAIRRO"].str.upper() == "TATUQUARA") & 
        (df["ENDERECO"].str.contains("BR", case=False, na=False)) & 
        (df["ENDERECO"].str.contains("116", case=False, na=False))
    )
    
    df.loc[mask, "LAT"] = lat_ceasa
    df.loc[mask, "LON"] = lon_ceasa

Coordenadas da CEASA (BR-116, 22881) aplicadas manualmente.


In [1124]:
# Coordenadas corretas
lat_affonso = -25.442899546484774
lon_affonso = -49.21377457314697

dataframes = [df_filtrado]

for df in dataframes:
    mask = (
        (df["ENDERECO"].str.upper() == "AVENIDA PRESIDENTE AFFONSO CAMARGO") &
        (df["CEP"].astype(str) == "80060090")
    )
    
    df.loc[mask, "LAT"] = lat_affonso
    df.loc[mask, "LON"] = lon_affonso

Coordenadas da Av. Presidente Affonso Camargo aplicadas manualmente.


In [189]:
def corrigir_enderecos(df):
    def ajustar(texto):
        if isinstance(texto, str):
            texto = re.sub(r"\bROD\.\s*", "RODOVIA ", texto)
            texto = re.sub(r"\bR\.\s*", "RUA ", texto)
            texto = re.sub(r"\bAV\.\s*", "AVENIDA ", texto)
            texto = re.sub(r"\bDR\.\s*", "DOUTOR ", texto)
            texto = re.sub(r"\bEST\.\s*", "ESTRADA ", texto)
            texto = re.sub(r"\DES\.\s*", "DESEMBARGADOR ", texto)
        return texto

    df["ENDERECO"] = df["ENDERECO"].apply(ajustar)
    return df

df_filtrado = corrigir_enderecos(df_filtrado)

In [1128]:
def corrigir_locais_especificos(row):
    endereco_str = str(row['ENDERECO']).upper()
    
    # 1. Caso CEASA
    if "CEASA" in endereco_str:
        row['ENDERECO'] = "RODOVIA REGIS BITTENCOURT (BR-116)"
        row['NUMERO'] = "22881"
        row['BAIRRO'] = "TATUQUARA"
        row['CEP'] = "81690901"
        
    # 2. Caso MERCADO MUNICIPAL
    elif "MERC" in endereco_str and "MUN" in endereco_str:
        row['ENDERECO'] = "AVENIDA SETE DE SETEMBRO"
        row['NUMERO'] = "1865"
        row['BAIRRO'] = "CENTRO"
        row['CEP'] = "80060070"

    # 3. Caso SHOPPING POPULAR
    elif "SHOPPING POPULAR" in endereco_str:
        row['ENDERECO'] = "RUA OTTO CABEL"
        row['NUMERO'] = "51"
        row['BAIRRO'] = "NOVO MUNDO"
        row['CEP'] = "81020245"

    
    return row

df_filtrado = df_filtrado.apply(corrigir_locais_especificos, axis=1)

In [1130]:
def limpeza_especifica_v2(row):
    end = str(row['ENDERECO']).upper()
    
    # Caso 1
    if "CARLOTA STRAUBE" in end:
        row['BAIRRO'] = "BOA VISTA"
        row['CEP'] = "82560330"
        
    # Caso 2
    if "COMERCIO SUL" in end:
        row['ENDERECO'] = "RUA COMERCIO SUL"
        row['BAIRRO'] = "CIDADE INDUSTRIAL"
        
    # Caso 3
    if "ERICO VERISSIMO" in end:
        row['ENDERECO'] = "RUA ERICO VERISSIMO"
        row['BAIRRO'] = "ALTO BOQUEIRAO"
        
    # Caso 4
    if "BR 116-KM 10" in end or "CEASA" in end:
        row['ENDERECO'] = "RODOVIA REGIS BITTENCOURT"
        row['NUMERO'] = "22881"
        row['BAIRRO'] = "TATUQUARA"

    
    return row

df_filtrado = df_filtrado.apply(limpeza_especifica_v2, axis=1)

In [191]:
df_filtrado["BAIRRO"] = df_filtrado["BAIRRO"].str.upper()

import unicodedata

# Função para remover acentos
def remover_acentos(texto):
    return ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )

colunas = [
    "ENDERECO",
    "NUMERO",
    "BAIRRO",
    "CEP"
]

for col in colunas:
    df_filtrado[col] = (
        df_filtrado[col]
        .astype(str)
        .str.strip()
        .str.upper()
        .apply(remover_acentos)
    )


df_filtrado["CEP"] = df_filtrado["CEP"].str.replace(r'\D', '', regex=True)

print(f"Total de linhas originais: {len(df_filtrado)}")

Total de linhas originais: 1667


In [193]:
import numpy as np

def limpar_campos(df):
    if "BAIRRO" in df.columns:
        df["CEP"] = df["CEP"].replace("***", np.nan)
    
    return df

df_filtrado = limpar_campos(df_filtrado)

In [1136]:
import pandas as pd
import re
import unicodedata


def sem_acento(txt):
    if pd.isna(txt):
        return ""
    txt = str(txt)
    txt = unicodedata.normalize("NFKD", txt)
    return "".join(c for c in txt if not unicodedata.combining(c))

def limpar_texto(s):
    if pd.isna(s):
        return ""

    s = str(s).strip().upper()

    # Corrigir encoding ruim
    s = s.replace("Âº", "º")
    s = s.replace("Ã", "A")

    # Ajustes comuns
    s = s.replace("1º", "PRIMEIRO")
    s = s.replace("H. ", "HERACLITO ")
    s = s.replace("I. ", "IGNACIO ")
    s = s.replace("DANDRADE", "D'ANDRADE")
    s = s.replace("ARCO-VERDE", "ARCOVERDE")

    # Remover marcacoes pouco úteis
    s = re.sub(r"\bLD\b", "", s)

    # Limpar pontuacao duplicada
    s = re.sub(r"\s+", " ", s).strip(" ,.-")

    return s


# =========================================================
# DICIONARIO DE AJUSTES ESPECIFICOS
# =========================================================
ajustes_endereco = {
    "RUA JULIA HUGA MARIA NEGRELLO": "RUA JULIA HUGA MARIA REBELLO",
    "RUA DOUTOR JOAQUIM DO AMARAL": "RUA JOAQUIM AMARAL",
    "JORNALISTA AUGUSTO WALDRIGUES": "RUA JORNALISTA AUGUSTO WALDRIGUES",
    "HAMILCAR PIZZATTO": "RUA HAMILCAR PIZZATTO",
    "RUA CAPITAO ZEPPIM" : "RUA CAPITAO ZEPPIN",
    "RIO IGUACU": "RUA RIO IGUACU",
    "RODOVIA BR 116": "BR-116",
    "RODOVIA CURITIBA PARANAGUA BR-277": "BR-277", 
    "RODOVIA CURITIBA RIO BRANCO PR-092": "PR-092",
    "RUA HUMBERTO HIGINO PAROLIM" : "RUA HUMBERTO HIGINO PAROLIN",
    "RUA JORNALISTA SILVINO A. BATISTA" : "RUA JORNALISTA SILVINO ALVES BATISTA", 
    "AVENIDA PRESIDENTE AFFONSO CAMARGO, CENTRO, CURITIBA, PR" : "AVENIDA PRESIDENTE AFFONSO CAMARGO, JARDIM BOTANICO, CURITIBA, PR",
    "RUA RUA" : "RUA", 
}

def aplicar_ajuste_endereco(s):
    if pd.isna(s):
        return s
    
    s = str(s)
    
    for original, novo in ajustes_endereco.items():
        if original in s:
            s = s.replace(original, novo)
    
    return s

df_filtrado["ENDERECO"] = (
    df_filtrado["ENDERECO"]
    .apply(limpar_texto)
    .apply(aplicar_ajuste_endereco)
    .apply(sem_acento)
)


df_filtrado["BAIRRO"] = (
    df_filtrado["BAIRRO"]
    .apply(limpar_texto)
    .apply(sem_acento)
)

In [195]:
df_filtrado["ENDERECO_COMPLETO"] = (
    df_filtrado["ENDERECO"].fillna("") + ", " +
    df_filtrado["NUMERO"].fillna("").astype(str) + ", " +
    df_filtrado["BAIRRO"].fillna("") + ", CURITIBA, PR, " +
    df_filtrado["CEP"].fillna("").astype(str)
)

# Limpar possíveis vírgulas duplicadas ou espaços
df_filtrado["ENDERECO_COMPLETO"] = (
    df_filtrado["ENDERECO_COMPLETO"]
    .str.replace(r",\s*,", ",", regex=True)
    .str.strip(", ")
)

In [202]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import os

# 1. Configuração do Geolocator
geolocator = Nominatim(user_agent="tcc_curitiba_final", timeout=15)
geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1.2,
    max_retries=3,
    error_wait_seconds=5
)

df_filtrado["LAT"] = None
df_filtrado["LON"] = None

# 2. Nome do arquivo de backup
ARQUIVO_BACKUP = "df_filtrado_2.csv"

print(f"Total de endereços para geocodificar: {len(df_filtrado)}")


# 3. Loop de geocodificação
for i, idx in enumerate(df_filtrado.index):
    try:
        endereco = df_filtrado.at[idx, "ENDERECO_COMPLETO"]
        location = geocode(endereco)

        if location:
            df_filtrado.at[idx, "LAT"] = location.latitude
            df_filtrado.at[idx, "LON"] = location.longitude

        # Salva progresso a cada 20 registros
        if (i + 1) % 20 == 0:
            df_filtrado.to_csv(ARQUIVO_BACKUP, index=False)
            print(f"Progresso: {i + 1}/{len(df_filtrado)} endereços processados")

    except Exception as e:
        print(f"Erro no índice {idx}: {e}")
        continue

# 4. Salvamento Final
df_filtrado.to_csv(ARQUIVO_BACKUP, index=False)
print("--- Geocodificação Finalizada com Sucesso! ---")

Total de endereços para geocodificar: 1667
Progresso: 20/1667 endereços processados
Progresso: 40/1667 endereços processados
Progresso: 60/1667 endereços processados
Progresso: 80/1667 endereços processados
Progresso: 100/1667 endereços processados
Progresso: 120/1667 endereços processados
Progresso: 140/1667 endereços processados
Progresso: 160/1667 endereços processados
Progresso: 180/1667 endereços processados
Progresso: 200/1667 endereços processados
Progresso: 220/1667 endereços processados
Progresso: 240/1667 endereços processados
Progresso: 260/1667 endereços processados
Progresso: 280/1667 endereços processados
Progresso: 300/1667 endereços processados
Progresso: 320/1667 endereços processados
Progresso: 340/1667 endereços processados
Progresso: 360/1667 endereços processados
Progresso: 380/1667 endereços processados
Progresso: 400/1667 endereços processados
Progresso: 420/1667 endereços processados
Progresso: 440/1667 endereços processados
Progresso: 460/1667 endereços process

RateLimiter caught an error, retrying (0/3 tries). Called with (*('RUA AUGUSTO STRESSER, 1916, HUGO LANGE, CURITIBA, PR, 80040345',), **{}).
Traceback (most recent call last):
  File "C:\Users\Adm\anaconda3\Lib\site-packages\urllib3\connection.py", line 196, in _new_conn
    sock = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Adm\anaconda3\Lib\site-packages\urllib3\util\connection.py", line 60, in create_connection
    for res in socket.getaddrinfo(host, port, family, socket.SOCK_STREAM):
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Adm\anaconda3\Lib\socket.py", line 964, in getaddrinfo
    for res in _socket.getaddrinfo(host, port, family, type, proto, flags):
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
socket.gaierror: [Errno 11001] getaddrinfo failed

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "C:

Progresso: 760/1667 endereços processados
Progresso: 780/1667 endereços processados
Progresso: 800/1667 endereços processados
Progresso: 820/1667 endereços processados
Progresso: 840/1667 endereços processados
Progresso: 860/1667 endereços processados
Progresso: 880/1667 endereços processados
Progresso: 900/1667 endereços processados
Progresso: 920/1667 endereços processados
Progresso: 940/1667 endereços processados
Progresso: 960/1667 endereços processados
Progresso: 980/1667 endereços processados


RateLimiter caught an error, retrying (0/3 tries). Called with (*('RUA FRANCISCO FRISCHMANN, 2597, PORTAO, CURITIBA, PR, 80320250',), **{}).
Traceback (most recent call last):
  File "C:\Users\Adm\anaconda3\Lib\site-packages\urllib3\connection.py", line 196, in _new_conn
    sock = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Adm\anaconda3\Lib\site-packages\urllib3\util\connection.py", line 60, in create_connection
    for res in socket.getaddrinfo(host, port, family, socket.SOCK_STREAM):
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Adm\anaconda3\Lib\socket.py", line 964, in getaddrinfo
    for res in _socket.getaddrinfo(host, port, family, type, proto, flags):
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
socket.gaierror: [Errno 11001] getaddrinfo failed

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "C:

Progresso: 1000/1667 endereços processados
Progresso: 1020/1667 endereços processados
Progresso: 1040/1667 endereços processados
Progresso: 1060/1667 endereços processados
Progresso: 1080/1667 endereços processados
Progresso: 1100/1667 endereços processados
Progresso: 1120/1667 endereços processados
Progresso: 1140/1667 endereços processados
Progresso: 1160/1667 endereços processados
Progresso: 1180/1667 endereços processados
Progresso: 1200/1667 endereços processados
Progresso: 1220/1667 endereços processados
Progresso: 1240/1667 endereços processados
Progresso: 1260/1667 endereços processados
Progresso: 1280/1667 endereços processados
Progresso: 1300/1667 endereços processados
Progresso: 1320/1667 endereços processados
Progresso: 1340/1667 endereços processados
Progresso: 1360/1667 endereços processados
Progresso: 1380/1667 endereços processados
Progresso: 1400/1667 endereços processados
Progresso: 1420/1667 endereços processados
Progresso: 1440/1667 endereços processados
Progresso: 

In [204]:
# Contagem total de valores nulos nas colunas de coordenadas
nulos_lat = df_filtrado["LAT"].isna().sum()

print(f"Endereços não encontrados (LAT): {nulos_lat}")
print(f"Sucesso: {len(df_filtrado) - nulos_lat} de {len(df_filtrado)} endereços.")

Endereços não encontrados (LAT): 126
Sucesso: 1541 de 1667 endereços.


In [208]:
with pd.option_context(
    'display.max_rows', None,
    'display.max_columns', None,
    'display.max_colwidth', None
):
    print(
        df_filtrado[df_filtrado["LAT"].isna()]
        [["ENDERECO_COMPLETO",  "ENDERECO", "NUMERO", "BAIRRO", "CEP"]]
    )

                                                                                    ENDERECO_COMPLETO  \
4069       RUA ENGENHEIRO JULIO CESAR DE SOUZA ARAUJO, 271, CIDADE INDUSTRIAL, CURITIBA, PR, 81290270   
18557                              RUA FRANCISCO DEROSSO, 375, CAMPO COMPRIDO, CURITIBA, PR, 81240380   
19785                             RUA DANIEL BRAMBILA, 235, CIDADE INDUSTRIAL, CURITIBA, PR, 81310180   
24835                      RUA PADRE JOSE LOPACINSKI, 1170, CIDADE INDUSTRIAL, CURITIBA, PR, 81280080   
27932                          SHOPPING POPULAR - CAPAO RASO, NAN, BAIRRO NAO INFORMADO, CURITIBA, PR   
30477                               ESTRADA DO GANCHINHO- LD, 5163, GANCHINHO, CURITIBA, PR, 81935552   
45886                             ROD BR-116-INT CEASA-PAV A, NAN, BAIRRO NAO INFORMADO, CURITIBA, PR   
48637                      MERC MUNIC CTBA-PV A -BOX 194/195, NAN, BAIRRO NAO INFORMADO, CURITIBA, PR   
54295                                            RODOVI

In [1172]:
# Substituir o endereço específico da BR-116 em todas as linhas do df_filtrado
endereco_antigo = "RODOVIA REGIS BITTENCOURT (BR-116)" 
endereco_novo = "BR-116"

# Fazer a substituição na coluna ENDERECO_SEM_BAIRRO
df_filtrado["ENDERECO"] = df_filtrado["ENDERECO"].str.replace(
    endereco_antigo, 
    endereco_novo, 
    regex=False
)

In [1174]:
df_filtrado["ENDERECO_SEM_BAIRRO"] = (
    df_filtrado["ENDERECO"].fillna("") + ", " +
    df_filtrado["NUMERO"].fillna("").astype(str) + ", CURITIBA, PR, " +
    df_filtrado["CEP"].fillna("").astype(str)
)

# Limpar possíveis vírgulas duplicadas ou espaços
df_filtrado["ENDERECO_SEM_BAIRRO"] = (
    df_filtrado["ENDERECO_SEM_BAIRRO"]
    .str.replace(r",\s*,", ",", regex=True)
    .str.strip(", ")
)

In [1178]:
geolocator = Nominatim(user_agent="tcc_curitiba_retry", timeout=15)
geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1.2,
    max_retries=3,
    error_wait_seconds=5
)

# 1. Selecionar apenas os registros com LAT nula
df_sem_lat = df_filtrado[df_filtrado["LAT"].isna()].copy()

print(f"Iniciando tentativa de recuperação para {len(df_sem_lat)} endereços com LAT nula...")

# 2. Aplicar geocodificação apenas nos registros sem LAT
if len(df_sem_lat) > 0:
    df_sem_lat["location"] = df_sem_lat["ENDERECO_SEM_BAIRRO"].apply(geocode)
    
    # 3. Extrair latitude e longitude
    df_sem_lat["nova_LAT"] = df_sem_lat["location"].apply(lambda loc: loc.latitude if loc else None)
    df_sem_lat["nova_LON"] = df_sem_lat["location"].apply(lambda loc: loc.longitude if loc else None)
    
    # 4. Atualizar o dataframe original apenas onde estava nulo
    indices_com_sucesso = df_sem_lat[df_sem_lat["nova_LAT"].notna()].index
    
    df_filtrado.loc[indices_com_sucesso, "LAT"] = df_sem_lat.loc[indices_com_sucesso, "nova_LAT"]
    df_filtrado.loc[indices_com_sucesso, "LON"] = df_sem_lat.loc[indices_com_sucesso, "nova_LON"]
    
    # 5. Verificar o sucesso da operação
    recuperados = len(indices_com_sucesso)
    total = len(df_sem_lat)
    print(f"Sucesso: {recuperados} de {total} endereços foram localizados nesta rodada.")
    
    # 6. Visualizar o que ainda restou como None (se houver)
    ainda_nulos = df_filtrado[df_filtrado["LAT"].isna()]
    if not ainda_nulos.empty:
        print(f"\nAinda restam {len(ainda_nulos)} endereços não localizados:")
        print(ainda_nulos[["ENDERECO", "NUMERO", "BAIRRO", "ENDERECO_SEM_BAIRRO"]])
else:
    print("Não há registros com LAT nula para processar.")

Iniciando tentativa de recuperação para 14 endereços com LAT nula...
Sucesso: 14 de 14 endereços foram localizados nesta rodada.


In [1184]:
import geopandas as gpd
from shapely.geometry import Point

# Criar coluna geom (geometry)
df_filtrado["geometry"] = df_filtrado.apply(
    lambda row: Point(row["LON"], row["LAT"]) if pd.notna(row["LAT"]) else None,
    axis=1
)

# Converter para GeoDataFrame
gdf = gpd.GeoDataFrame(df_filtrado, geometry="geometry", crs="EPSG:4326")

In [1188]:
# Salvar como GeoPackage 
gdf.to_file('dados_filtrados.gpkg', layer='dados_para_intersect', driver='GPKG')

GeoPackage salvo! Use 'ogr2ogr' ou QGIS para importar no PostGIS


In [1190]:
import geopandas as gpd

# Salvar como shapefile
gdf.to_file("dados_filtrados_em_shapefile.shp", driver="ESRI Shapefile", encoding="utf-8")

C:\Users\Adm\AppData\Local\Temp\ipykernel_3648\1167033337.py:4: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file("dados_filtrados_em_shapefile.shp", driver="ESRI Shapefile", encoding="utf-8")
C:\Users\Adm\anaconda3\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'NOME_EMPRESARIAL' to 'NOME_EMPRE'
  ogr_write(
C:\Users\Adm\anaconda3\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'INICIO_ATIVIDADE' to 'INICIO_ATI'
  ogr_write(
C:\Users\Adm\anaconda3\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'NUMERO_DO_ALVARA' to 'NUMERO_DO_'
  ogr_write(
C:\Users\Adm\anaconda3\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'NOME_FANTASIA' to 'NOME_FANTA'
  ogr_write(
C:\Users\Adm\anaconda3\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'DAT

Shapefile salvo com sucesso!


In [1198]:
import pandas as pd

# Converte para DataFrame
df_comum = pd.DataFrame(df_filtrado.drop(columns='geometry'))

In [1200]:
df_comum.to_csv('alvaras_filtrados_geocodificados.csv', index=False)